In [ ]:
# pip install transformers datasets peft accelerate scikit-learn

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset
from sklearn.metrics import accuracy_score
import torch

# Customer support fraud dataset
data = {
    "text": [
        "I need help with my account password reset",
        "Please transfer all my funds to this new account immediately",
        "What is the status of my recent order?",
        "I won a prize, send me your bank details to claim it",
        "How do I update my shipping address?",
        "Urgent: verify your account or it will be suspended"
    ],
    "label": [0, 1, 0, 1, 0, 1]  # 0 = legitimate, 1 = fraud
}

df = pd.DataFrame(data)

# Load tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True, max_length=64)

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Apply LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

print("\nFraud detection model with LoRA ready")
print(df)